# Lab 1 — Ratings Data: Preprocessing + EDA
**ACM Workshop · Recommender Systems · Day 1 · 09:45–10:45 (60 min)**

**Goal of this lab:** understand the dataset the recommender will be built from.
Not "learn Pandas" — answer three questions:

1. **Who are my users?**  2. **What are my items?**  3. **What does the rating data look like?**

**You will produce:** dataset statistics · data-quality report · rating distribution · user activity · item popularity · cold-start counts · user–item matrix · sparsity · a clean train/test split saved to `data/processed/` (reused in Labs 2 and 3).

> Pipeline position: **Raw data → `EDA` → similarity → CF → MF → prediction → top-N → evaluation**

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20, "display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# --- locate (or download) MovieLens "latest-small" -----------------------------------------
URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
def find_data():
    for base in ["data", "../data", "../../data", "../../../data"]:
        p = Path(base) / "ml-latest-small"
        if (p / "ratings.csv").exists():
            return p
    Path("data").mkdir(exist_ok=True)
    print("Downloading MovieLens latest-small (~1 MB)...")
    urllib.request.urlretrieve(URL, "data/ml.zip")
    zipfile.ZipFile("data/ml.zip").extractall("data")
    return Path("data/ml-latest-small")

DATA = find_data()
PROC = DATA.parent / "processed"; PROC.mkdir(exist_ok=True)
FIG  = DATA.parent.parent / "figures"; FIG.mkdir(exist_ok=True)
print("data folder :", DATA.resolve())

## 1. Load the data
MovieLens *latest-small* (GroupLens, University of Minnesota): ~100 k ratings from 610 users on ~9.7 k movies, collected 1996–2018.
Each user has rated **at least 20 movies** — GroupLens already filtered the data this way.

In [ ]:
ratings = pd.read_csv(DATA / "ratings.csv")
movies  = pd.read_csv(DATA / "movies.csv")
ratings.head()

In [ ]:
print("ratings:", ratings.shape, "| movies:", movies.shape)
ratings.info()

In [ ]:
movies.head()

In [ ]:
n_users, n_items, n_ratings = ratings.userId.nunique(), ratings.movieId.nunique(), len(ratings)
print(f"users   : {n_users:,}")
print(f"movies  : {n_items:,}  (catalogue has {len(movies):,}; some are never rated)")
print(f"ratings : {n_ratings:,}")

**Concept check.** What do the columns mean?
`userId` = who · `movieId` = what · `rating` = how much (explicit feedback, 0.5–5.0 in half-star steps) · `timestamp` = when (Unix seconds).
Titles and genres live in a *separate* table → we will need a `merge`.

## 2. Data quality

In [ ]:
print("missing values per column:\n", ratings.isnull().sum(), "\n")
print("fully duplicated rows        :", ratings.duplicated().sum())
print("duplicate (user, movie) pairs:", ratings.duplicated(["userId", "movieId"]).sum())
ratings.describe().round(2)

**Questions** — answer in a markdown cell:
- Missing values? Duplicates? → *None, this dataset is already clean.*
- Min / max / mean rating?
- Why do we check **(user, movie) duplicates** specifically? *Because the matrix cell `R[u, i]` must hold exactly one value.*
- Do all ratings in `movies.csv` have a matching movie? Check below.

In [ ]:
orphan = set(ratings.movieId) - set(movies.movieId)
unrated = set(movies.movieId) - set(ratings.movieId)
print("rated movies missing from movies.csv:", len(orphan))
print("catalogue movies nobody has rated   :", len(unrated))

## 3. Rating distribution
Before modelling, look at what a rating *is* in this data.

In [ ]:
counts = ratings.rating.value_counts().sort_index()
print(counts)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].bar(counts.index.astype(str), counts.values, color="#2b6cb0")
ax[0].set(title="Rating frequency", xlabel="rating", ylabel="# ratings")
ax[1].bar(["mean", "median"], [ratings.rating.mean(), ratings.rating.median()], color="#dd6b20")
ax[1].set(title=f"Mean = {ratings.rating.mean():.2f}", ylim=(0, 5))
plt.tight_layout(); plt.savefig(FIG / "lab1_rating_dist.png"); plt.show()

**Observations to discuss**
- Ratings are **skewed positive**: 4.0 is the most common value, and whole stars are more common than half stars.
- People mostly rate things they *chose to watch* → **selection bias**. A rating of 3.5 is therefore not "average movie", it is "average among movies someone picked".
- The global mean (~3.5) is already a decent predictor. **Any model must beat it** (we will use it as a baseline in Labs 2–3).

## 4. User analysis — who are my users?

In [ ]:
per_user = ratings.groupby("userId").size().rename("n_ratings")
print(per_user.describe().round(1))
print("\nTop-5 most active users:\n", per_user.nlargest(5))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(per_user, bins=60, color="#2b6cb0"); ax[0].set(title="Ratings per user", xlabel="# ratings", ylabel="# users")
ax[1].hist(np.log10(per_user), bins=40, color="#2b6cb0"); ax[1].set(title="Same, log10 x-axis", xlabel="log10(# ratings)")
plt.tight_layout(); plt.savefig(FIG / "lab1_user_activity.png"); plt.show()

share = per_user.sort_values(ascending=False).cumsum() / per_user.sum()
print(f"Top 10% of users ({int(.1*len(per_user))}) produce {share.iloc[int(.1*len(per_user))-1]:.0%} of all ratings")

In [ ]:
# Do heavy raters rate differently? Mean rating per user.
user_mean = ratings.groupby("userId").rating.mean()
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.hist(user_mean, bins=30, color="#38a169"); ax.set(title="Average rating given by each user", xlabel="user mean rating", ylabel="# users")
plt.tight_layout(); plt.savefig(FIG / "lab1_user_mean.png"); plt.show()
print(f"user means range from {user_mean.min():.2f} to {user_mean.max():.2f}")

**Take-aways**
- The distribution is **heavy-tailed**: a few "power users" rate thousands of movies, the typical user rates ~70 (median).
- Users have different **rating habits** — some are generous (mean 4.5), some harsh (mean 2.5). This is called **user bias**, and is why Lab 2 *mean-centres* ratings and why SVD adds a bias term $b_u$.

## 5. Item analysis — what are my items?

In [ ]:
per_item = ratings.groupby("movieId").size().rename("n_ratings")
item_stats = (ratings.groupby("movieId").rating.agg(["mean", "count"])
              .join(movies.set_index("movieId")[["title", "genres"]]))
print(per_item.describe().round(1))
print("\nMost-rated movies:")
item_stats.nlargest(8, "count")[["title", "count", "mean"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
s = per_item.sort_values(ascending=False).values
ax[0].plot(s, color="#c53030"); ax[0].set(title="Long tail: ratings per movie (ranked)", xlabel="movie rank", ylabel="# ratings")
ax[1].hist(per_item.clip(upper=50), bins=50, color="#c53030"); ax[1].set(title="Ratings per movie (clipped at 50)", xlabel="# ratings")
plt.tight_layout(); plt.savefig(FIG / "lab1_long_tail.png"); plt.show()

In [ ]:
# Careful: "highest rated" without a minimum count is misleading.
print("Top-rated, NO minimum count:")
display(item_stats.nlargest(5, "mean")[["title", "count", "mean"]])
print("Top-rated, at least 50 ratings:")
item_stats[item_stats["count"] >= 50].nlargest(5, "mean")[["title", "count", "mean"]]

**Take-aways**
- The **long tail**: a handful of blockbusters get hundreds of ratings; most movies get very few.
- Movies with 1–2 ratings have **unreliable averages** (a single 5★ = "best movie ever"). Every recommender needs a *support threshold* or *shrinkage*.
- This is the root of **popularity bias**: popular movies have more data, so models know them better and recommend them more.

## 6. Cold start — quantify it
> *What happens when a new user joins, or a new movie has no ratings?*

In [ ]:
for k in [1, 5, 10, 20]:
    print(f"movies with <= {k:>2} ratings: {(per_item <= k).sum():>5,}  ({(per_item <= k).mean():.1%} of rated movies)")
print(f"\nmovies in the catalogue with ZERO ratings: {len(movies) - len(per_item)}")
print(f"users with < 20 ratings: {(per_user < 20).sum()}  (dataset guarantees >= 20 — real systems do not!)")

**Discuss (2 min, pairs):** A collaborative-filtering model can only recommend things that *someone has rated*. What should Netflix do for (a) a brand-new user, (b) a brand-new film? *(Hints: popularity, content features, asking onboarding questions → Day 2 / hybrids.)*

## 7. Genres & time (bonus context)

In [ ]:
movies["year"] = movies.title.str.extract(r"\((\d{4})\)\s*$").astype(float)
movies["genre_list"] = movies.genres.str.split("|")
g = (ratings.merge(movies[["movieId", "genre_list"]], on="movieId")
            .explode("genre_list").groupby("genre_list").rating.agg(["count", "mean"]).sort_values("count", ascending=False))
g.round(2).head(12)

In [ ]:
ratings["dt"] = pd.to_datetime(ratings.timestamp, unit="s")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ratings.dt.dt.year.value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#805ad5"); ax[0].set(title="Ratings per year", xlabel="")
g["count"].head(12)[::-1].plot(kind="barh", ax=ax[1], color="#805ad5"); ax[1].set(title="Ratings per genre", ylabel="")
plt.tight_layout(); plt.savefig(FIG / "lab1_genre_time.png"); plt.show()
print("date range:", ratings.dt.min().date(), "→", ratings.dt.max().date())

## 8. The user–item matrix and sparsity  ⭐ *the key concept of the lab*
Rows = users, columns = movies, values = ratings, **NaN = unknown**.

In [ ]:
user_item = ratings.pivot_table(index="userId", columns="movieId", values="rating")
print("matrix shape:", user_item.shape)
cells = user_item.shape[0] * user_item.shape[1]
print(f"possible cells : {cells:,}")
print(f"observed cells : {user_item.notna().sum().sum():,}")
print(f"sparsity       : {1 - user_item.notna().sum().sum()/cells:.2%} of the matrix is EMPTY")
user_item.iloc[:8, :8]

In [ ]:
# Visualise: densest corner of the matrix vs a random corner
top_u = per_user.nlargest(120).index; top_i = per_item.nlargest(120).index
rng = np.random.default_rng(0)
rnd_u = rng.choice(user_item.index, 120, replace=False); rnd_i = rng.choice(user_item.columns, 120, replace=False)
fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
for a, (u, i, t) in zip(ax, [(top_u, top_i, "120 most active users × 120 most popular movies"),
                              (rnd_u, rnd_i, "120 random users × 120 random movies")]):
    sub = user_item.loc[u, i].notna().values
    a.imshow(sub, cmap="Blues", aspect="auto", interpolation="nearest")
    a.set(title=f"{t}\n{sub.mean():.1%} filled", xticks=[], yticks=[])
plt.tight_layout(); plt.savefig(FIG / "lab1_sparsity.png"); plt.show()

**Why is it so empty?** Nobody can watch 9,000 movies. Even the most engaged users have seen a tiny fraction of the catalogue.
A recommender's job is exactly this: **predict the empty cells** — and then rank them.

> *This is the conceptual bridge to the next session: how can we predict a missing cell using other users' behaviour?*

## 9. Pre-processing decisions & the train/test split
Decisions we make (and why):

| Decision | Choice | Reason |
|---|---|---|
| Duplicates / NaNs | none to remove | checked in §2 |
| Drop rare movies? | **No** for the split; Lab 2 applies a *support threshold at recommendation time* | dropping from the data changes the evaluation problem |
| Rating scale | keep 0.5–5.0 | Surprise `Reader(rating_scale=(0.5, 5))` |
| Split | random 80 / 20 with `random_state=42`, **same split in all labs** | comparable RMSE numbers |

> ⚠️ Random splits are the simplest option. A *temporal* split (train on past, test on future) is more realistic but gives different numbers — an extension for the curious.

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(ratings[["userId", "movieId", "rating"]], test_size=0.2, random_state=42)
train.to_csv(PROC / "train.csv", index=False); test.to_csv(PROC / "test.csv", index=False)
ratings[["userId", "movieId", "rating"]].to_csv(PROC / "ratings_clean.csv", index=False)
movies[["movieId", "title", "genres", "year"]].to_csv(PROC / "movies_clean.csv", index=False)

print(f"train: {len(train):,}   test: {len(test):,}")
# How many test ratings involve a user / movie never seen in training? (a mini cold-start measurement)
print("test rows whose user  is unseen in train:", (~test.userId.isin(train.userId)).sum())
print("test rows whose movie is unseen in train:", (~test.movieId.isin(train.movieId)).sum(),
      f"({(~test.movieId.isin(train.movieId)).mean():.1%})")

## 10. Exercises (finish in the last 10 min)

**E1.** Which genre has the highest *mean* rating among genres with ≥ 1,000 ratings? Which has the lowest?
**E2.** Find the 5 users whose mean rating is lowest (min. 50 ratings). How would that affect a naive "average of neighbours" prediction?
**E3.** What fraction of all ratings goes to the 100 most-rated movies?

In [ ]:
# E1
# TODO: write your code here


In [ ]:
# E2
# TODO: write your code here


In [ ]:
# E3
# TODO: write your code here


## ✅ Lab 1 deliverable — checklist
- [x] dataset statistics · [x] missing-value / duplicate check · [x] rating distribution
- [x] user activity · [x] item popularity (long tail) · [x] cold-start counts
- [x] user–item matrix + sparsity · [x] saved `data/processed/{train,test}.csv`

**Next (11:00): Collaborative Filtering theory** — *"How can we fill those empty cells using other users' behaviour?"*